# Classical Computer Vision Preprocessing

This script applies OpenCV-based preprocessing to the raw guitar dataset
before training the Improved CNN. It performs:

1. **Background reduction** via contour detection (crop main guitar region)
2. **Histogram equalisation** (CLAHE) for lighting consistency
3. **Edge enhancement** via an unsharp mask sharpening filter

Input:  `dataset/`           (acoustic/, electric/, bass/)
Output: `processed_dataset/` (acoustic/, electric/, bass/)

## 1. Imports

In [1]:
import os
import cv2
import numpy as np
from pathlib import Path

## 2. Configuration

In [2]:
RAW_DIR       = "dataset"            # Source ImageFolder root
PROCESSED_DIR = "processed_dataset" # Output ImageFolder root
IMAGE_SIZE    = 224                  # Final output size (pixels)
CLASSES       = ["acoustic", "electric", "bass"]

# CLAHE parameters
CLAHE_CLIP    = 2.0   # Contrast limit
CLAHE_TILE    = (8, 8) # Tile grid size

# Contour crop: minimum fraction of image area for a valid guitar contour
MIN_CONTOUR_AREA_RATIO = 0.05

## 3. Preprocessing Functions

In [3]:
def find_guitar_crop(bgr_img: np.ndarray) -> np.ndarray:
    """
    Background reduction via contour detection.

    Steps:
        1. Convert to grayscale.
        2. Gaussian blur to suppress noise.
        3. Otsu's adaptive thresholding.
        4. Find contours → pick the largest one.
        5. If large enough, crop the bounding rectangle; else return original.

    Args:
        bgr_img: Input image in BGR colour space (OpenCV default).

    Returns:
        Cropped BGR image containing the main guitar region.
    """
    h, w = bgr_img.shape[:2]
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)

    # Blur before threshold to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Otsu's binarisation – automatically finds optimal threshold
    _, thresh = cv2.threshold(blurred, 0, 255,
                               cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Morphological closing fills small holes in the guitar silhouette
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    # Find external contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return bgr_img  # Nothing found → return original

    # Pick the largest contour by area
    largest = max(contours, key=cv2.contourArea)
    area_ratio = cv2.contourArea(largest) / (h * w)

    if area_ratio < MIN_CONTOUR_AREA_RATIO:
        return bgr_img  # Contour too small (likely noise) → skip crop

    # Crop the bounding rectangle with a small padding
    x, y, cw, ch = cv2.boundingRect(largest)
    pad = 10
    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(w, x + cw + pad)
    y2 = min(h, y + ch + pad)

    return bgr_img[y1:y2, x1:x2]


def apply_clahe(bgr_img: np.ndarray) -> np.ndarray:
    """
    Apply CLAHE (Contrast Limited Adaptive Histogram Equalisation) in LAB space.

    Operating in LAB keeps colour information intact while improving lightness
    channel contrast, avoiding the colour artefacts of equalising RGB directly.

    Args:
        bgr_img: Input BGR image.

    Returns:
        BGR image with enhanced contrast.
    """
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l_channel, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
    l_eq = clahe.apply(l_channel)

    lab_eq = cv2.merge((l_eq, a, b))
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)


def unsharp_mask(bgr_img: np.ndarray,
                 sigma: float = 1.0,
                 strength: float = 1.5) -> np.ndarray:
    """
    Edge enhancement via unsharp masking.

    Formula: sharpened = original + strength * (original - blurred)

    This emphasises high-frequency detail (edges) without introducing
    the ringing artefacts of pure Laplacian sharpening.

    Args:
        bgr_img:  Input BGR image.
        sigma:    Gaussian blur radius (controls edge scale).
        strength: How strongly edges are enhanced.

    Returns:
        Sharpened BGR image.
    """
    blurred  = cv2.GaussianBlur(bgr_img, (0, 0), sigma)
    sharpened = cv2.addWeighted(bgr_img, 1 + strength,
                                 blurred, -strength, 0)
    return sharpened


def preprocess_image(img_path: str) -> np.ndarray | None:
    """
    Full preprocessing pipeline for a single image.

    Pipeline:
        Read → Background crop → CLAHE → Sharpen → Resize

    Args:
        img_path: Absolute or relative path to the source image.

    Returns:
        Preprocessed BGR image (IMAGE_SIZE × IMAGE_SIZE) or None on failure.
    """
    bgr = cv2.imread(img_path)
    if bgr is None:
        return None  # Unreadable file (corrupt, unsupported format, etc.)

    # Step 1: Background reduction – crop to guitar bounding box
    bgr = find_guitar_crop(bgr)

    # Step 2: Histogram equalisation (CLAHE) in LAB colour space
    bgr = apply_clahe(bgr)

    # Step 3: Unsharp mask edge enhancement
    bgr = unsharp_mask(bgr)

    # Step 4: Resize to target size
    bgr = cv2.resize(bgr, (IMAGE_SIZE, IMAGE_SIZE),
                      interpolation=cv2.INTER_AREA)

    return bgr

## 4. Process the Full Dataset

In [4]:
def preprocess_dataset(src_root: str = RAW_DIR,
                        dst_root: str = PROCESSED_DIR) -> None:
    """
    Walk every class folder in src_root, apply preprocessing to each image,
    and write the results to the corresponding folder in dst_root.

    Args:
        src_root: Path to the raw ImageFolder dataset.
        dst_root: Path to the output processed dataset.
    """
    src_path = Path(src_root)
    dst_path = Path(dst_root)

    if not src_path.is_dir():
        raise FileNotFoundError(
            f"[ERROR] Source dataset not found at '{src_root}'. "
            "Please run dataset collection first."
        )

    supported = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    total_ok  = 0
    total_err = 0

    for cls_dir in sorted(src_path.iterdir()):
        if not cls_dir.is_dir():
            continue

        out_dir = dst_path / cls_dir.name
        out_dir.mkdir(parents=True, exist_ok=True)

        img_files = [f for f in cls_dir.iterdir()
                     if f.suffix.lower() in supported]

        print(f"\n[INFO] Processing class '{cls_dir.name}' "
              f"({len(img_files)} images) …")

        for img_file in img_files:
            processed = preprocess_image(str(img_file))

            if processed is None:
                print(f"  [SKIP] {img_file.name} — could not be read")
                total_err += 1
                continue

            out_path = out_dir / img_file.name
            cv2.imwrite(str(out_path), processed)
            total_ok += 1

        print(f"  → Saved {len(img_files) - total_err} images to {out_dir}")

    print(f"\n{'='*60}")
    print(f"  Preprocessing complete.")
    print(f"  Successfully processed : {total_ok}")
    print(f"  Skipped (errors)       : {total_err}")
    print(f"  Output saved to        : '{dst_root}/'")
    print(f"{'='*60}")


# Run the preprocessing pipeline
preprocess_dataset()


[INFO] Processing class 'acoustic' (1524 images) …
  → Saved 1524 images to processed_dataset/acoustic

[INFO] Processing class 'bass' (1368 images) …
  → Saved 1368 images to processed_dataset/bass

[INFO] Processing class 'electric' (1543 images) …
  → Saved 1543 images to processed_dataset/electric

  Preprocessing complete.
  Successfully processed : 4435
  Skipped (errors)       : 0
  Output saved to        : 'processed_dataset/'
